In [8]:
import pandas as pd
import numpy as np
import ConsultasBD
import MuestreoEInterpolacion
import FiltroButterworth
import MediaMovil
import Metricas
#importlib.reload(ConsultasBD, MuestreoEInterpolacion)

In [11]:
#Datos del usuario que se quiere reprocesar
usuarioId = 12
numeroPruebas = 1
fmuestreo = 100
fcorte = 10
ventanaMediaMovil = 100 # la ventana es de 100 muestras que equivale a un segundo

#Nombres de las tablas de bases de datos
DB_LecturasBalanceBoard = 'LecturasBalanceBoard'
DB_LecturasInterpoladas = 'LecturasInterpoladas'
DB_LecturasConFiltro = 'LecturasConFiltro'
DB_LecturasConMediaMovil = 'LecturasConMediaMovil'
DB_Evaluaciones = 'Evaluaciones'

In [12]:
def EliminarPrimerYUltimoSegundo(df, ventana):
    # Eliminar la primeras 'ventana' filas (por NaN). Estas tienen Nan por aplicar la media móvil
    # si la media móvil es de 100 muestras, que equivalen a 1 segundos, entonces las primeras 100 muestras
    # tienen valor Nan
    df = df.dropna().reset_index(drop=True)

    #df = df.iloc[:ventana].reset_index(drop=True) #Puede que con esto se soluciones
    df = df.iloc[:-ventana].reset_index(drop=True)
    print(f"✅ Media móvil aplicada. Se eliminaron el primer y último segundo ({ventana} muestras cada uno).")
    print(f"Total de muestras resultantes: {len(df)}")
    return df

def AnadirIntervaloDeTiempoEnSegundos(df):
    df["TimeStampSegundos"] = (
        (df["TimeStamp"] - df["TimeStamp"].iloc[0]).dt.total_seconds()
    )
    return df

if __name__ == "__main__":
    #Procesamiento de datos
    dfDatosReales = ConsultasBD.ObtenerLecturasBalanceDeBD(usuarioId, numeroPruebas)
    dfDatosMuestreados = MuestreoEInterpolacion.Interpolar(dfDatosReales, fs = fmuestreo)
    dfDatosConFiltro = FiltroButterworth.Aplicar_filtro_butterworth(dfDatosMuestreados, fc = fcorte, fs = fmuestreo)
    dfDatosMediaMovil = MediaMovil.Aplicar_media_movil(dfDatosConFiltro, ventana = ventanaMediaMovil)

    #Se añade una nueva columna con un intervalo de tiempos en segundos
    dfDatosMuestreados = AnadirIntervaloDeTiempoEnSegundos(dfDatosMuestreados)
    dfDatosConFiltro = AnadirIntervaloDeTiempoEnSegundos(dfDatosConFiltro)
    dfDatosMediaMovil = AnadirIntervaloDeTiempoEnSegundos(dfDatosMediaMovil)

    #Calcular métricas y eliminar el primer y último segundo
    dfDatosMediaMovil = EliminarPrimerYUltimoSegundo(dfDatosMediaMovil, ventana = ventanaMediaMovil)
    metricas = Metricas.Calcular_metricas(dfDatosMediaMovil, usuarioId=usuarioId, numeroPruebas=numeroPruebas)

    # Guardar cada etapa en su tabla correspondiente
    ConsultasBD.Insertar_dataframe(dfDatosMuestreados, DB_LecturasInterpoladas, usuarioId, numeroPruebas)
    ConsultasBD.Insertar_dataframe(dfDatosConFiltro, DB_LecturasConFiltro, usuarioId, numeroPruebas)
    ConsultasBD.Insertar_dataframe(dfDatosMediaMovil, DB_LecturasConMediaMovil, usuarioId, numeroPruebas)
    ConsultasBD.Actualizar_tiempos_balanceboard(dfDatosReales, DB_LecturasBalanceBoard)

    # Guardar métricas finales
    ConsultasBD.Insertar_metricas(metricas)

c:\Users\alexs\Desktop\Preprocesamiento\ConsultasBD.py:21: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query_ConsultaLecturasBalance, conn, parse_dates=['TimeStamp'])


Filas leídas: 3112
Primeras 10 filas:
      Id  UsuarioId  NumeroPruebas    TopLeft  TopRight  BottomLeft  \
0  26021         12              1  -0.074480 -0.385269    6.033200   
1  26022         12              1   7.522453  2.311615    7.745850   
2  26023         12              1   7.522453  2.311615    7.745850   
3  26024         12              1  13.890471  3.968272   11.210074   
4  26025         12              1  13.890471  3.968272   11.210074   
5  26026         12              1  10.315444  2.427196   11.482541   
6  26027         12              1   5.399781  0.577904   12.338866   
7  26028         12              1   5.399781  0.577904   12.338866   
8  26029         12              1   3.537787 -0.038527   14.440756   
9  26030         12              1   5.474261  0.500850   18.722382   

   BottomRight     COP_X     COP_Y     Total               TimeStamp  
0    -1.728814 -0.451448 -0.148700  0.961159 2025-11-16 17:17:28.393  
1    -0.998870 -0.180956  0.022342  4.

# PRUEBAS DE CÓDIGO

In [4]:
import numpy as np
x = np.array([-1, -3, 5, 7, -1])
np.diff(x)

array([-2,  8,  2, -8])

In [5]:
import pandas as pd

# 1. Crear un DataFrame a partir de un diccionario
datos = {
    'columna1': [-1, -2, -3, -4, -5],
    'columna2': [-10, 20, -30, 40, -50],
    'columna3': [100, 200, 300, 400, 500]
}
df = pd.DataFrame(datos)

# Mostrar el DataFrame
print("DataFrame creado:")
print(df)

# 2. Calcular la media de todas las columnas
medias_columnas = df.mean()
print("\nMedia de cada columna:")
print(medias_columnas)

# 3. Calcular la media de una columna específica
media_columna1 = df['columna1'].mean()
print(f"\nMedia de 'columna1': {media_columna1}")

# 4. Calcular la media de las filas (axis=1)
medias_filas = df.mean(axis=1)
print("\nMedia de cada fila:")
print(medias_filas)


DataFrame creado:
   columna1  columna2  columna3
0        -1       -10       100
1        -2        20       200
2        -3       -30       300
3        -4        40       400
4        -5       -50       500

Media de cada columna:
columna1     -3.0
columna2     -6.0
columna3    300.0
dtype: float64

Media de 'columna1': -3.0

Media de cada fila:
0     29.666667
1     72.666667
2     89.000000
3    145.333333
4    148.333333
dtype: float64
